# Notebook 4: Out-of-Sample Backtest & Quant Performance Report

Trong notebook này, chúng ta thực hiện kiểm định (Backtest) chiến lược trên dữ liệu hoàn toàn chưa biết (Out-of-Sample từ `2023-07-01` đến `2024-06-30`):
1. Mô phỏng tái cơ cấu danh mục hàng tuần/hàng kỳ với chi phí giao dịch và thuế (`0.20%/trade`).
2. Tính toán bảng KPI tài chính định lượng chuyên nghiệp: **Sharpe Ratio**, **Maximum Drawdown**, **CAGR**, **Information Ratio**.
3. Trực quan hóa biểu đồ chuẩn xuất bản: Equity Curve, Underwater Drawdowns, Rolling Beta, và Allocation History.


In [26]:
!git clone https://github.com/PTN2004/AI-Driven-Market-Neutral-Portfolio-Optimization.git
%cd AI-Driven-Market-Neutral-Portfolio-Optimization
!pip install -qr requirements.txt

Cloning into 'AI-Driven-Market-Neutral-Portfolio-Optimization'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 337 (delta 158), reused 328 (delta 149), pack-reused 0 (from 0)
Receiving objects: 100% (337/337), 7.39 MiB | 26.66 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/AI-Driven-Market-Neutral-Portfolio-Optimization/AI-Driven-Market-Neutral-Portfolio-Optimization


In [27]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.models import create_dataloaders, AlphaMLP, AlphaTrainer, AlphaPredictor
from src.risk_models import RiskModel, BetaCalculator
from src.optimization import PortfolioOptimizer
from src.backtest import BacktestEngine, PerformanceEvaluator, BacktestVisualizer

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Chuẩn bị Pipeline và Mô hình AI đã huấn luyện


In [28]:
fetcher = DataFetcher()
raw_data = fetcher.fetch_all_group()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

master_df = preprocessor.prepare_tabular_dataset(normalized_data, Config.START_DATE, Config.END_DATE)
train_loader, val_loader, test_loader, test_df = create_dataloaders(master_df, batch_size=64)

model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
trainer = AlphaTrainer(model, lr=1e-3)
history = trainer.fit(train_loader, val_loader, epochs=15)

predictor = AlphaPredictor(model)
test_df_with_preds = predictor.predict_all(test_df)
for sym in normalized_data.keys():
    if sym == Config.BENCHMARK_TICKER:
        continue
    sym_preds = test_df_with_preds[test_df_with_preds['symbol'] == sym].set_index('date')['predicted_mu']
    normalized_data[sym] = normalized_data[sym].merge(sym_preds, on='date', how='left')
    normalized_data[sym]['predicted_mu'] = normalized_data[sym]['predicted_mu'].ffill().fillna(0.0)


[2026-08-04 04:43:46] [INFO] [DataFetcher]: Get group symbol ['ACB', 'ANV', 'BAF', 'BCM', 'BID', 'BMP', 'BSI', 'BSR', 'BVH', 'BWE', 'CII', 'CMG', 'CTD', 'CTG', 'CTR', 'CTS', 'DBC', 'DCM', 'DGW', 'DIG', 'DPM', 'DSE', 'DXG', 'EIB', 'EVF', 'FPT', 'FRT', 'FTS', 'GAS', 'GEE', 'GEX', 'GMD', 'GVR', 'HAG', 'HCM', 'HDB', 'HDG', 'HHV', 'HPG', 'HSG', 'HT1', 'KBC', 'KDC', 'KDH', 'KOS', 'LPB', 'MBB', 'MCH', 'MSB', 'MSN', 'MWG', 'NAB', 'NKG', 'NLG', 'NT2', 'NVL', 'OCB', 'PAN', 'PC1', 'PDR', 'PHR', 'PLX', 'PNJ', 'POW', 'PVD', 'PVT', 'REE', 'SAB', 'SBT', 'SHB', 'SIP', 'SJS', 'SSB', 'SSI', 'STB', 'TAL', 'TCB', 'TCH', 'TCX', 'TPB', 'VCB', 'VCG', 'VCI', 'VCK', 'VGC', 'VHC', 'VHM', 'VIB', 'VIC', 'VIX', 'VJC', 'VND', 'VNM', 'VPB', 'VPI', 'VPL', 'VPX', 'VRE', 'VSC', 'VTP']


INFO:DataFetcher:Get group symbol ['ACB', 'ANV', 'BAF', 'BCM', 'BID', 'BMP', 'BSI', 'BSR', 'BVH', 'BWE', 'CII', 'CMG', 'CTD', 'CTG', 'CTR', 'CTS', 'DBC', 'DCM', 'DGW', 'DIG', 'DPM', 'DSE', 'DXG', 'EIB', 'EVF', 'FPT', 'FRT', 'FTS', 'GAS', 'GEE', 'GEX', 'GMD', 'GVR', 'HAG', 'HCM', 'HDB', 'HDG', 'HHV', 'HPG', 'HSG', 'HT1', 'KBC', 'KDC', 'KDH', 'KOS', 'LPB', 'MBB', 'MCH', 'MSB', 'MSN', 'MWG', 'NAB', 'NKG', 'NLG', 'NT2', 'NVL', 'OCB', 'PAN', 'PC1', 'PDR', 'PHR', 'PLX', 'PNJ', 'POW', 'PVD', 'PVT', 'REE', 'SAB', 'SBT', 'SHB', 'SIP', 'SJS', 'SSB', 'SSI', 'STB', 'TAL', 'TCB', 'TCH', 'TCX', 'TPB', 'VCB', 'VCG', 'VCI', 'VCK', 'VGC', 'VHC', 'VHM', 'VIB', 'VIC', 'VIX', 'VJC', 'VND', 'VNM', 'VPB', 'VPI', 'VPL', 'VPX', 'VRE', 'VSC', 'VTP']


[2026-08-04 04:43:46] [INFO] [DataFetcher]: Starting data ingestion for 101 symbols (2021-01-01 -> 2025-12-30)...


INFO:DataFetcher:Starting data ingestion for 101 symbols (2021-01-01 -> 2025-12-30)...


[2026-08-04 04:43:46] [INFO] [DataFetcher]: fetching data symbol VNINDEX


...

INFO:AlphaTrainer:Epoch 10/15 - Train Loss: 0.7066 - Val Loss: 0.7838 - Val IC: 0.0833


[2026-08-04 04:45:24] [INFO] [AlphaTrainer]: Epoch 15/15 - Train Loss: 0.6931 - Val Loss: 0.8009 - Val IC: 0.0488


INFO:AlphaTrainer:Epoch 15/15 - Train Loss: 0.6931 - Val Loss: 0.8009 - Val IC: 0.0488


[2026-08-04 04:45:24] [INFO] [AlphaTrainer]: Training completed.


INFO:AlphaTrainer:Training completed.


In [29]:
trainer.save_model("best_w.pt")

In [30]:
import os
len(os.listdir("/content/AI-Driven-Market-Neutral-Portfolio-Optimization/data/raw/"))

106

## 2. Thực thi Backtest Engine Out-of-Sample (Tái cơ cấu Hàng tuần)


In [31]:
from typing import Dict, Any
import numpy as np
import pandas as pd

from src.utils.config import Config
from src.utils.logger import get_logger

logger = get_logger("BacktestEngine")

class BacktestEngine:
    """
    Out-of-sample backtesting engine simulation with realistic transaction fees,
    rebalancing frequency, and slippage modeling.
    """
    def __init__(
        self,
        predictor: Any,
        risk_model: Any,
        beta_calc: Any,
        optimizer: Any,
        rebalance_freq: str = "weekly",
        fee_rate: float = 0.0020
    ):
        self.predictor = predictor
        self.risk_model = risk_model
        self.beta_calc = beta_calc
        self.optimizer = optimizer
        self.rebalance_freq = rebalance_freq
        self.fee_rate = fee_rate
        self.benchmark = Config.BENCHMARK_TICKER

    def _should_rebalance(self, date: pd.Timestamp, step_idx: int) -> bool:
        if step_idx == 0:
            return True
        if self.rebalance_freq == "weekly":
            return date.dayofweek == 0 # Monday
        elif self.rebalance_freq == "monthly":
            return date.day <= 5 and step_idx % 20 == 0
        return step_idx % 5 == 0

    def run(
        self,
        cleaned_data: Dict[str, pd.DataFrame],
        normalized_data: Dict[str, pd.DataFrame],
        start_date: str = Config.TEST_START_DATE,
        end_date: str = Config.END_DATE
    ) -> Dict[str, pd.DataFrame]:
        logger.info(f"Starting Out-of-Sample backtest ({start_date} -> {end_date})...")
        
        bench_df = cleaned_data[self.benchmark]
        bench_sub = bench_df[(bench_df['date'] >= start_date) & (bench_df['date'] <= end_date)].copy()
        dates = pd.to_datetime(bench_sub['date']).values

        symbols = [s for s in cleaned_data.keys() if s != self.benchmark]
        
        # Pre-extract daily price matrix for speed
        price_matrix = {}
        for sym in symbols:
            df = cleaned_data[sym]
            sub = df[(df['date'] >= start_date) & (df['date'] <= end_date)].set_index('date')['close']
            price_matrix[sym] = sub
        price_df = pd.DataFrame(price_matrix).ffill().bfill()

        current_weights = pd.Series(0.0, index=symbols)
        results = []
        weights_history = []

        for idx, date_dt in enumerate(dates):
            date_str = pd.to_datetime(date_dt).strftime("%Y-%m-%d")
            rebalanced = False
            tx_cost = 0.0

            # 1. Calculate daily portfolio asset returns using YESTERDAY's weights
            if idx > 0 and date_str in price_df.index and price_df.index[idx-1] in price_df.index:
                p_today = price_df.iloc[idx]
                p_prev = price_df.iloc[idx - 1]
                daily_ret_vec = (p_today - p_prev) / (p_prev + 1e-8)
                port_ret = float((current_weights * daily_ret_vec).sum())
            else:
                port_ret = 0.0

            # 2. Rebalance at the end of TODAY using today's closing data
            if self._should_rebalance(pd.to_datetime(date_str), idx):
                try:
                    # 2.1 Predict Alpha
                    mu_dict = self.predictor.predict_for_date(normalized_data, date_str)
                    mu_series = pd.Series(mu_dict)
                    for s in symbols:
                        if s not in mu_series:
                            mu_series[s] = 0.0
                    mu_series = mu_series.loc[symbols].fillna(0.0)

                    # 2.2 Risk Modeling
                    cov_mat = self.risk_model.compute_covariance(cleaned_data, date_str)
                    betas = self.beta_calc.compute_betas(cleaned_data, date_str)

                    # 2.3 Optimize Portfolio
                    new_weights = self.optimizer.optimize(mu_series, cov_mat, betas)
                    new_weights = new_weights.reindex(symbols).fillna(0.0)

                    # Calculate transaction cost from weight turnover
                    turnover = (new_weights - current_weights).abs().sum()
                    tx_cost = turnover * self.fee_rate
                    
                    # Deduct transaction cost from today's portfolio return
                    port_ret -= float(tx_cost)

                    current_weights = new_weights
                    rebalanced = True
                except Exception as e:
                    logger.debug(f"Rebalance skipped on {date_str}: {e}")

            # 3. Benchmark return
            bench_ret = 0.0
            if idx > 0:
                b_today = bench_sub.iloc[idx]['close']
                b_prev = bench_sub.iloc[idx - 1]['close']
                bench_ret = (b_today - b_prev) / (b_prev + 1e-8)

            # Portfolio stats
            gross_exp = current_weights.abs().sum()
            net_exp = current_weights.sum()
            try:
                betas_today = self.beta_calc.compute_betas(cleaned_data, date_str).reindex(symbols).fillna(1.0)
                port_beta = (current_weights * betas_today).sum()
            except Exception:
                port_beta = 0.0

            results.append({
                "date": date_str,
                "portfolio_return": port_ret,
                "benchmark_return": bench_ret,
                "turnover_cost": tx_cost,
                "gross_exposure": gross_exp,
                "net_exposure": net_exp,
                "portfolio_beta": port_beta,
                "rebalanced": rebalanced
            })

            w_row = current_weights.to_dict()
            w_row["date"] = date_str
            weights_history.append(w_row)

        results_df = pd.DataFrame(results)
        weights_df = pd.DataFrame(weights_history)
        logger.info("Out-of-Sample backtest completed successfully.")
        return {"results_df": results_df, "weights_df": weights_df}


In [32]:
engine = BacktestEngine(
    predictor=predictor,
    risk_model=RiskModel(),
    beta_calc=BetaCalculator(),
    optimizer=PortfolioOptimizer(),
    rebalance_freq='weekly',
    fee_rate=0.0020
)

backtest_res = engine.run(cleaned_data, normalized_data, start_date=Config.TEST_START_DATE, end_date=Config.END_DATE)
results_df = backtest_res['results_df']
weights_df = backtest_res['weights_df']
print('Hoàn tất Backtest Out-of-Sample!')


[2026-08-04 04:45:26] [INFO] [BacktestEngine]: Starting Out-of-Sample backtest (2025-07-01 -> 2025-12-30)...


INFO:BacktestEngine:Starting Out-of-Sample backtest (2025-07-01 -> 2025-12-30)...


[2026-08-04 04:45:53] [INFO] [BacktestEngine]: Out-of-Sample backtest completed successfully.


INFO:BacktestEngine:Out-of-Sample backtest completed successfully.


Hoàn tất Backtest Out-of-Sample!


## 3. Báo cáo Tóm tắt Hiệu suất (KPI Summary Table)


In [33]:
evaluator = PerformanceEvaluator()
summary_df = evaluator.evaluate(results_df)
display(summary_df)


[2026-08-04 04:45:53] [INFO] [PerformanceEvaluator]: Calculating quantitative performance metrics...


INFO:PerformanceEvaluator:Calculating quantitative performance metrics...


,AI Market Neutral,VN-Index (Buy & Hold),Equal-Weight VN100
Cumulative Return,0.00%,27.36%,24.71%
Annualized Return (CAGR),0.00%,60.99%,54.47%
Annualized Volatility,0.00%,20.83%,16.67%
Sharpe Ratio,0.00,2.25,2.51
Max Drawdown (MDD),0.00%,-10.54%,-8.17%
Daily Win Rate,0.00%,63.28%,64.06%
Information Ratio,-2.39,0.00,-0.67


## 4. Biểu đồ Đường cong Tài sản & Quản trị Rủi ro


In [34]:
Config.DATA_DIR = "content/drive/MyDrive"
print(Config.DATA_DIR)

content/drive/MyDrive


In [35]:
visualizer = BacktestVisualizer(output_dir=Path('notebook_charts'))

# 1. Equity Curves
visualizer.plot_equity_curves(results_df)
plt.show()

# 2. Drawdowns
visualizer.plot_drawdowns(results_df)
plt.show()

# 3. Rolling Beta Verification
visualizer.plot_rolling_beta(results_df)
plt.show()

# 4. Long/Short Exposure
visualizer.plot_exposure_history(weights_df)
plt.show()
